This notebook reads the files the [search development plan](../notes/2026-11-11-search-development-plan.qmd) produces in `data/search_tests/` and shows the outcomes of its three steps. Step 1 compares the sources @chawinga2019 included with the RDS-Review criteria. Step 2 lists the terms those sources use for each part of the PCC table. Step 3 tests queries built from those terms in PubMed, Scopus and Web of Science. The notebook writes nothing.

`summary_of_outcomes.md` states the outcomes in prose, and `log.md` records the judgments behind the coding. The checks come first. If any check finds an error, the notebook stops.

In [ ]:
import math
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
T = ROOT / "data" / "search_tests"
DATABASES = {"PubMed": "pubmed", "Scopus": "scopus", "Web of Science": "web-of-science"}
PARTS = ["sharing", "reuse", "factors that obstruct", "factors that facilitate", "stakeholder perspectives",
         "policies and what drive them", "mechanisms for implementation"]


def read(name):
    return pd.read_csv(T / name, dtype=str, keep_default_na=False)


steps_1_2 = ["reference_candidates.csv", "reference_sources.csv", "reference_codes.csv", "counts.csv",
             "verdict.csv", "comparison.csv", "statement_matches.csv", "terms.csv", "term_syntax.csv"]
missing = [n for n in steps_1_2 if not (T / n).exists()]
have_data = not missing
if not have_data:
    print("Not run yet. Missing: " + ", ".join(missing))
else:
    cand, src, codes, counts, verdict, comparison, statements, terms, syntax = (read(n) for n in steps_1_2)
have_tests = (T / "query_tests.csv").exists()

## Checks

In [ ]:
if have_data:
    errors = []
    keys = set(src["key"])
    if set(cand["key"]) != keys:
        errors.append(f"reference_sources.csv and reference_candidates.csv list different sources: {sorted(set(cand['key']) ^ keys)}.")
    unknown = set(codes["key"]) - keys
    if unknown:
        errors.append(f"reference_codes.csv has keys not in reference_sources.csv: {sorted(unknown)}.")
    for f in ("Practice", "data_origin"):
        lacking = keys - set(codes.loc[codes["field"] == f, "key"])
        if lacking:
            errors.append(f"{len(lacking)} source(s) have no {f} code: {sorted(lacking)}.")
    pairs = set(zip(syntax["term"], syntax["type"], syntax["database"]))
    unwritten = [t for t in zip(terms["term"], terms["type"], terms["database"]) if t not in pairs]
    if unwritten:
        errors.append(f"{len(unwritten)} row(s) in terms.csv have no written form in term_syntax.csv: {unwritten[:5]}.")
    if errors:
        raise ValueError("\n".join(errors))
    print("All checks passed.")

## Step 1. Comparability assessment

The reference set holds the references @chawinga2019 cite as findings in sections 4 and 5 and Table 2. Sources first published from 2007 to 2014 form the development set, which step 2 draws terms from. Sources first published from 2015 to July 2018 form the test set, which step 3 measures coverage against.

In [ ]:
if have_data:
    sets = pd.crosstab(src["set"], src["eligible_apart_from_year"], margins=True, margins_name="total")
    sets.columns.name = "eligible apart from year"
    display(sets)
    reasons = (src.assign(reason=src["reason"].replace("", "(none)"))
               .groupby(["reason", "set"]).size().unstack(fill_value=0))
    display(reasons)

### Codes

Each count is the number of sources with that code. `codes_without_passage` counts the codes that rest on a title alone.

In [ ]:
if have_data:
    display(counts[counts["field"].isin(["Practice", "data_origin", "Factors"])].reset_index(drop=True))
    display(counts[~counts["field"].isin(["Practice", "data_origin", "Factors"])].reset_index(drop=True))

### Criteria compared

In [ ]:
if have_data:
    display(comparison)

### Search statements

The four search statements @chawinga2019 report, matched against the titles and recorded passages of the reference set.

In [ ]:
if have_data:
    display(statements.drop(columns=["keys"]))

### Verdict

A PCC part is *covered* when sources eligible apart from year carry it in both sets.

In [ ]:
if have_data:
    display(verdict)

## Step 2. Terms by PCC part

Terms come from the development set: the terms recorded in the coding, MeSH headings, author keywords and OpenAlex keywords. *Sharing* and *reuse* terms go to step 3. Factor terms go to screening.

In [ ]:
if have_data:
    n = terms.groupby(["pcc_part", "database"]).size().unstack(fill_value=0).reindex(PARTS)
    display(n)

In [ ]:
if have_data:
    sr = terms[(terms["database"] == "PubMed") & terms["pcc_part"].isin(["sharing", "reuse"])].copy()
    sr["count"] = sr["count"].astype(int)
    display(sr.sort_values(["pcc_part", "count"], ascending=[True, False])[["pcc_part", "term", "type", "count"]].reset_index(drop=True))

In [ ]:
if have_data:
    fac = terms[(terms["database"] == "PubMed") & ~terms["pcc_part"].isin(["sharing", "reuse"])]
    display(fac[["pcc_part", "term", "type", "count", "sources_added"]].reset_index(drop=True))

## Step 3. Query tests

Each database gets one query, built by the yield rule in the plan and tested for coverage of the test set. Precision comes from a random sample of 100 records per database, screened on title and abstract.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return round(p, 3), round(max(0, c - h), 3), round(min(1, c + h), 3)


if not have_tests:
    print("Not run yet. Missing: query_tests.csv")
else:
    tests = read("query_tests.csv")
    display(tests.set_index("database").drop(columns=["final_query"]).T)
    for _, t in tests.iterrows():
        print(f"{t['database']}: {t['final_query']}")

In [ ]:
if have_tests:
    sr = terms[terms["pcc_part"].isin(["sharing", "reuse"])]
    status = sr.groupby(["database", "status"]).size().unstack(fill_value=0)
    display(status)
    inq = sr[sr["status"].isin(["in query", "fallback in query"])]
    display(inq[["database", "pcc_part", "term", "type", "development_yield", "load_2015_2026", "sources_added"]].reset_index(drop=True))

### Misses

Indexed test set sources the query misses, and development set sources reached only by terms excluded by yield.

In [ ]:
if have_tests and (T / "query_misses.csv").exists():
    display(read("query_misses.csv"))

### Precision

Precision counts *undecidable* records once as included and once as left out.

In [ ]:
if have_tests:
    rows = []
    for db, slug in DATABASES.items():
        p = T / f"precision_sample_{slug}.csv"
        if not p.exists():
            continue
        s = read(p.name)
        d = s["decision"].str.lower()
        if (d == "").any():
            print(f"{db}: {int((d == '').sum())} of {len(s)} sampled records not screened yet.")
            continue
        inc, exc, und = int((d == "include").sum()), int((d == "exclude").sum()), int((d == "undecidable").sum())
        a, b = wilson(inc + und, len(s)), wilson(inc, inc + exc)
        rows.append(dict(database=db, sampled=len(s), included=inc, excluded=exc, undecidable=und,
                         title_only=int((s["abstract_source"] == "none").sum()),
                         precision_undecidable_included=a[0], ci_included=f"{a[1]}-{a[2]}",
                         precision_undecidable_left_out=b[0], ci_left_out=f"{b[1]}-{b[2]}"))
    if rows:
        display(pd.DataFrame(rows).set_index("database"))